In [4]:
#@title Imports
##########################################################
# Always include all imports at the first executable cell.
##########################################################
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod # Abstract Base Classes for Python

from sklearn.datasets import fetch_openml
from keras.utils import to_categorical
import matplotlib.image as mpimg
import seaborn as sns; sns.set()
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from pathlib import Path
import tensorflow as tf

In [5]:
DATA_PATH = Path('/Users/edward/Documents/GWU/ml/kaggle/nfl-big-data-bowl-2026-prediction')
TRAIN_PATH = DATA_PATH / 'train'
WEEKS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

inputs, outputs = [], []
for w in WEEKS:
  inp = pd.read_csv(TRAIN_PATH / f'input_2023_w{w:02d}.csv')
  out = pd.read_csv(TRAIN_PATH / f'output_2023_w{w:02d}.csv')
  inputs.append(inp)
  outputs.append(out)
# print(inputs[0])
# print(outputs[0])
input = pd.concat(inputs, ignore_index=True)
output = pd.concat(outputs, ignore_index=True)


print(f"Data loaded: {len(input)} input rows, {input['game_id'].nunique()} games")
print(f"Data loaded: {len(output)} output rows, {output['game_id'].nunique()} games")
# print(input.columns)

Data loaded: 3236116 input rows, 180 games
Data loaded: 368514 output rows, 180 games


In [6]:
def create_features(df):
    df = df.copy()
    
    # Parse player height to feet
    def height_to_feet(h):
        # Return NaN for missing values
        if pd.isna(h):
            return np.nan

        # Split the string by the '-' delimiter
        parts = str(h).split('-')

        # Ensure the format is correct (e.g., 'feet-inches')
        if len(parts) == 2:
            try:
                feet = float(parts[0])
                inches = float(parts[1])

                # 1 foot = 12 inches, 1 inch = 2.54 cm
                total_inches = (feet * 12) + inches
                cm = total_inches * 2.54
                return cm
            except ValueError:
                # Handle cases where parts are not valid numbers
                return np.nan

        # Return NaN if the format is incorrect
        return np.nan
    
    
    #Player's Height in centimeters
    df['player_height_cm'] = df['player_height'].apply(height_to_feet)
    df.drop('player_height', axis=1, inplace=True)
    
    # Direction
    df['dir_rad'] = np.deg2rad(df['dir'])
    df['o_rad'] = np.deg2rad(df['o'])
    
    # Velocity components
    df['velocity_x'] = df['s'] * np.cos(df['dir_rad'])
    df['velocity_y'] = df['s'] * np.sin(df['dir_rad'])
    
    # Acceleration components
    df['acceleration_x'] = df['a'] * np.cos(df['dir_rad'])
    df['acceleration_y'] = df['a'] * np.sin(df['dir_rad'])
    
    # Momentum
    df['momentum_x'] = df['velocity_x'] * df['player_weight']
    df['momentum_y'] = df['velocity_y'] * df['player_weight']
    
    # Kinetic energy
    df['kinetic_energy'] = 0.5 * df['player_weight'] * (df['s'] ** 2)
    
    # Role indicators
    df['is_offense'] = (df['player_side'] == 'Offense').astype(int)
    df['is_defense'] = (df['player_side'] == 'Defense').astype(int)
    df['is_receiver'] = (df['player_role'] == 'Targeted Receiver').astype(int)
    df['is_coverage'] = (df['player_role'] == 'Defensive Coverage').astype(int)
    df['is_passer'] = (df['player_role'] == 'Passer').astype(int)
    
    # Ball features
    df['distance_to_ball'] = np.sqrt((df['x'] - df['ball_land_x'])**2 + (df['y'] - df['ball_land_y'])**2)
    df['angle_to_ball'] = np.arctan2(df['ball_land_y'] - df['y'], df['ball_land_x'] - df['x'])
    df['ball_direction_x'] = df['ball_land_x'] - df['x']
    df['ball_direction_y'] = df['ball_land_y'] - df['y']
    
    # Closing speed
    df['velocity_toward_ball'] = (df['velocity_x'] * df['ball_direction_x'] + 
                                   df['velocity_y'] * df['ball_direction_y']) / (df['distance_to_ball'] + 1e-6)
    df['closing_speed'] = df['velocity_toward_ball']
    
    return df

input_df = create_features(input)
print(f"Features created: {input_df.shape[1]} columns")
print(input_df.columns)
print(input_df['play_direction'])

modified_df = input_df.copy()
modified_df.drop('s', axis=1, inplace=True)
modified_df.drop('a', axis=1, inplace=True)
modified_df.drop('dir', axis=1, inplace=True)
print(input_df['o'])

sample_df = input_df.sample(n=5)
print(sample_df)

Features created: 43 columns
Index(['game_id', 'play_id', 'player_to_predict', 'nfl_id', 'frame_id',
       'play_direction', 'absolute_yardline_number', 'player_name',
       'player_weight', 'player_birth_date', 'player_position', 'player_side',
       'player_role', 'x', 'y', 's', 'a', 'dir', 'o', 'num_frames_output',
       'ball_land_x', 'ball_land_y', 'player_height_cm', 'dir_rad', 'o_rad',
       'velocity_x', 'velocity_y', 'acceleration_x', 'acceleration_y',
       'momentum_x', 'momentum_y', 'kinetic_energy', 'is_offense',
       'is_defense', 'is_receiver', 'is_coverage', 'is_passer',
       'distance_to_ball', 'angle_to_ball', 'ball_direction_x',
       'ball_direction_y', 'velocity_toward_ball', 'closing_speed'],
      dtype='object')
0          right
1          right
2          right
3          right
4          right
           ...  
3236111    right
3236112    right
3236113    right
3236114    right
3236115    right
Name: play_direction, Length: 3236116, dtype: object
0  

In [7]:
# Test prediction with ensemble (average of CV folds)
test_input = pd.read_csv(DATA_PATH / 'test_input.csv')
test_template = pd.read_csv(DATA_PATH / 'test.csv')

test_input = create_features(test_input)

In [8]:
#@title Define a base class class for Learning Algorithm

class BaseLearningAlgorithm(ABC):
  """Base class for a Supervised Learning Algorithm."""

  @abstractmethod
  def fit(self, x_train:np.array, y_train: np.array
          , x_val:np.array
          , y_val:np.array) -> None:
    """Trains a model from labels y and examples X.
    We include validation set for optional hyperparameter tuning. Not
    all of the algorithims we use will
    """

  @abstractmethod
  def predict(self, x_test: np.array) -> np.array:
    """Predicts on an unlabeled sample, X."""

  @property
  @abstractmethod
  def name(self) -> str:
    """Returns the name of the algorithm."""

In [9]:
def train_eval(learning_algo: BaseLearningAlgorithm, x_train, y_train,x_val, y_val, x_test, y_test):
  """Trains and evaluates the generic model."""
  learning_algo.fit(x_train, y_train, x_val, y_val)
  y_pred = learning_algo.predict(x_test)
  mat = confusion_matrix(y_test, y_pred)
  sns.set(rc = {'figure.figsize':(8,8)})
  sns.heatmap(mat.T, square=True, annot=True, fmt='d', cbar=False,
              xticklabels=['%d' %i for i in range(10)],
              yticklabels=['%d' %i for i in range(10)])
  plt.xlabel('true label')
  plt.ylabel('predicted label')
  plt.title(learning_algo.name)